# Train Analysis And Mode Selection

This notebook evaluates the current SVG gate against `train.csv`, compares `body_only` and `full_svg`, and writes a recommendation for the batch notebooks.


## Drive And Repo Setup

Mount Drive, clone the repo into the runtime, and copy the CSV inputs into the checkout.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output root
- Rerun-safe: Yes. It reclones the repo and recopies the inputs cleanly.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "svg-kaggle-comptetition" / "submission_outputs"

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    if destination.exists():
        continue

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    copied = False
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            copied = True
            break
    if not copied:
        for candidate in DRIVE_ROOT.rglob(required_name):
            if candidate.is_file():
                shutil.copy2(candidate, destination)
                copied = True
                break
    if not copied:
        raise FileNotFoundError(
            f"Could not find {required_name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
        )

os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)
sys.path.insert(0, str(CHECKOUT_PATH))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Drive output root: {OUTPUT_ROOT}")


## Package Check

Install any missing runtime packages required by the shared helper module and this notebook.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs missing packages.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "cairosvg": "cairosvg",
    "skimage": "scikit-image",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Load Helpers And Tables

Import the shared helper module, resolve analysis paths, and load the train/test/sample tables for later cells.

- Reads: Repo checkout, helper module, competition CSV files
- Writes: In-memory tables only
- Rerun-safe: Yes. It only reloads notebook state.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

import submission_pipeline as pipeline

pipeline.set_global_seed(1337)
PATHS = pipeline.resolve_pipeline_paths(Path(os.environ["SVG_KAGGLE_REPO_ROOT"]), OUTPUT_ROOT)
CONFIG = pipeline.GenerationConfig(verbose_progress=True)
TEST_DF, SAMPLE_SUBMISSION_DF, TRAIN_DF = pipeline.load_competition_frames(PATHS)

print(f"Repo root: {PATHS.repo_root}")
print(f"Output root: {PATHS.output_root}")

FIXED_STRATEGY = "deterministic_plus_repair"
SMOKE_TEST_ROWS = 10


## Reference Gate Audit

Run the current validator against every reference SVG in `train.csv` and save the detailed audit artifact.

- Reads: `train.csv`, current SVG validator
- Writes: `analysis/train_reference_gate_audit.csv`
- Rerun-safe: Yes. It recomputes and overwrites the audit artifact.


In [ ]:
REFERENCE_GATE_AUDIT_DF = pipeline.audit_reference_gate(TRAIN_DF)
REFERENCE_AUDIT_SUMMARY_DF, REFERENCE_AUDIT_ERRORS_DF = pipeline.summarize_reference_gate_audit(REFERENCE_GATE_AUDIT_DF)
REFERENCE_GATE_AUDIT_DF.to_csv(PATHS.analysis_dir / "train_reference_gate_audit.csv", index=False)
display(REFERENCE_AUDIT_SUMMARY_DF)
display(REFERENCE_AUDIT_ERRORS_DF)


## Load Model

Load the base model plus the LoRA adapter once so the smoke-test cells can reuse the same runtime state.

- Reads: Base model on Hugging Face, LoRA adapter from repo
- Writes: In-memory model state only
- Rerun-safe: No. It is safe to rerun, but it reloads the model and costs time.


In [ ]:
RUNTIME = pipeline.load_lora_runtime(PATHS.adapter_dir)
print(f"Loaded runtime on: {RUNTIME.runtime_device}")


## Smoke-Test Row Selection

Pick 10 train rows for the smoke test, prioritizing reference SVGs that already pass the current gate.

- Reads: `train.csv`, reference gate audit results
- Writes: In-memory smoke-test selection only
- Rerun-safe: Yes. It is deterministic because the seed is fixed.


In [ ]:
SMOKE_TEST_DF = pipeline.select_smoke_test_rows(
    TRAIN_DF,
    REFERENCE_GATE_AUDIT_DF,
    rows=SMOKE_TEST_ROWS,
    seed=1337,
)
display(SMOKE_TEST_DF[["id", "prompt", "reference_gate_valid"]])


## Body-Only Smoke Test

Generate SVGs for the smoke-test rows using `body_only` mode and the fixed sampled-repair strategy.

- Reads: Smoke-test rows, model runtime
- Writes: In-memory results only
- Rerun-safe: Yes. It is safe to rerun, but it recomputes the body-only outputs.


In [ ]:
BODY_ONLY_RESULTS_DF = pipeline.run_smoke_test(
    SMOKE_TEST_DF,
    RUNTIME,
    generation_mode="body_only",
    allow_sampled_repair=True,
    config=CONFIG,
)
display(pipeline.summarize_smoke_test(BODY_ONLY_RESULTS_DF))


## Full-SVG Smoke Test

Generate SVGs for the same smoke-test rows using `full_svg` mode and the same fixed sampled-repair strategy.

- Reads: Smoke-test rows, model runtime
- Writes: In-memory results only
- Rerun-safe: Yes. It is safe to rerun, but it recomputes the full-SVG outputs.


In [ ]:
FULL_SVG_RESULTS_DF = pipeline.run_smoke_test(
    SMOKE_TEST_DF,
    RUNTIME,
    generation_mode="full_svg",
    allow_sampled_repair=True,
    config=CONFIG,
)
display(pipeline.summarize_smoke_test(FULL_SVG_RESULTS_DF))


## Aggregate Comparison And Recommendation

Combine the smoke-test outputs, pick the winning generation mode, and save the analysis artifacts for downstream notebooks.

- Reads: Smoke-test outputs, analysis directory
- Writes: `analysis/train_smoke_test_results.csv`, `analysis/analysis_recommendation.json`
- Rerun-safe: Yes. It recomputes and overwrites the analysis artifacts.


In [ ]:
SMOKE_RESULTS_DF = pipeline.normalize_results_df(
    pd.concat([BODY_ONLY_RESULTS_DF, FULL_SVG_RESULTS_DF], ignore_index=True)
)
SMOKE_SUMMARY_DF = pipeline.summarize_smoke_test(SMOKE_RESULTS_DF)
RECOMMENDED_GENERATION_MODE = pipeline.choose_generation_mode(SMOKE_SUMMARY_DF)
pipeline.save_analysis_artifacts(
    PATHS,
    audit_df=REFERENCE_GATE_AUDIT_DF,
    smoke_results_df=SMOKE_RESULTS_DF,
    smoke_summary_df=SMOKE_SUMMARY_DF,
    recommended_generation_mode=RECOMMENDED_GENERATION_MODE,
    fixed_strategy=FIXED_STRATEGY,
)
print(f"Recommended generation mode: {RECOMMENDED_GENERATION_MODE}")
display(SMOKE_SUMMARY_DF)


## Inspection View

Display the reference SVG, body-only output, and full-SVG output for each smoke-test row so you can inspect failures visually.

- Reads: Reference SVGs, smoke-test result tables
- Writes: Nothing
- Rerun-safe: Yes. Read-only display cell.


In [ ]:
for row in SMOKE_TEST_DF.itertuples(index=False):
    display(Markdown(f"### Train row `{row.id}`\n**Prompt:** {row.prompt}"))
    if bool(row.reference_gate_valid):
        display(Markdown("**Reference SVG**"))
        display(SVG(data=row.svg))
    else:
        print("Reference SVG does not pass the current gate; surrogate scores for this row are informational only.")

    for mode_name, results_df in [("body_only", BODY_ONLY_RESULTS_DF), ("full_svg", FULL_SVG_RESULTS_DF)]:
        candidate = results_df[results_df["id"] == row.id].iloc[0]
        display(Markdown(f"**{mode_name}**"))
        print(f"selected_strategy: {candidate['selected_strategy']}")
        print(f"valid_final: {candidate['valid_final']}")
        print(f"gate_error: {candidate['gate_error']}")
        print(f"runtime_seconds: {candidate['runtime_seconds']:.2f}")
        raw_snippet = candidate['raw_text'].replace("\n", " ")[:300]
        print(f"raw_text snippet: {raw_snippet}")
        if candidate['clean_svg']:
            display(SVG(data=candidate['clean_svg']))
        else:
            print("No valid SVG produced.")
